# CSDR API key auth from Google Colab

This notebook is a minimal example of using a CSDR API key from Google Colab.

Before running it in Colab:
1. Open the Colab secrets panel.
2. Add a secret named `csdr-cloud-spatial-app`.
3. Paste your CSDR API key as the secret value.

The notebook then uses that key to call `GET /api/v0/dataset` and confirms the request succeeds.

In [ ]:
import os

import pandas as pd
import requests
from IPython.display import display

try:
    from google.colab import userdata
except ImportError:
    userdata = None

In [ ]:
APP_BASE_URL = os.getenv("CSDR_APP_BASE_URL", "https://csdr.dev.oceandevelopmentdata.org")
API_BASE_URL = f"{APP_BASE_URL}/api/v0"

if userdata is not None:
    API_KEY = userdata.get("csdr-cloud-spatial-app")
else:
    API_KEY = os.getenv("CSDR_CLOUD_SPATIAL_APP_API_KEY") or os.getenv("CSDR_API_KEY")

if not API_KEY:
    raise OSError(
        "API key not found. In Colab, add a secret named 'csdr-cloud-spatial-app'. "
        "Outside Colab, set CSDR_CLOUD_SPATIAL_APP_API_KEY or CSDR_API_KEY."
    )

session = requests.Session()
session.headers.update(
    {
        "Accept": "application/json",
        "x-api-key": API_KEY,
    }
)


def api_get(path: str, params: dict | None = None) -> dict:
    url = f"{API_BASE_URL.rstrip('/')}/{path.lstrip('/')}"
    response = session.get(url, params=params, timeout=60)
    response.raise_for_status()

    payload = response.json()
    if "data" not in payload:
        raise ValueError(f"API response missing 'data' field: {response.text}")

    return payload["data"]


print("API base:", API_BASE_URL)
print("API key loaded successfully")

In [ ]:
dataset_list = api_get("/dataset", params={"page": 1, "size": 20})
datasets_df = pd.DataFrame(dataset_list["data"])

print(f"Dataset query succeeded. Returned {len(datasets_df)} dataset rows.")
display(datasets_df[["id", "name", "mainRunId", "createdAt"]])